## IT Department Management Staffing Analysis (Flag 84)

### Dataset Overview
This dataset contains 500 simulated records from the ServiceNow asset/sys_user table. Columns include sys_updated_on, serial_number, purchased_on, asset_tag, warranty_expiration, assigned_to, cost, model_category, display_name, and ci. Model categories include Computer, Printer, Rack, Storage Device, Computer Peripheral, Server, and Web Server. The average asset cost is $3,013.

### Your Objective
**Objective**: Evaluate the distribution of assets across model categories and analyze cost patterns to identify imbalances that may lead to budget inefficiencies.

**Role**: IT Asset Manager

**Category**: Asset Management

### Import Necessary Libraries
This cell imports all necessary libraries required for the analysis. This includes libraries for data manipulation, data visualization, and any specific utilities needed for the tasks.


In [1]:
import argparse
import pandas as pd
import json
import requests
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pandas import date_range

### Load User Agent Dataset
This cell loads user agent dataset used in the analysis. The dataset is stored in a CSV file and is loaded into a DataFrame. This step includes reading the data from a file path and possibly performing initial observations such as viewing the first few rows to ensure it has loaded correctly.


In [2]:
import pandas as pd
dataset_path = "csvs/flag-84.csv"
flag_data = pd.read_csv(dataset_path)
df = pd.read_csv(dataset_path)
flag_data.head()

,category,state,closed_at,opened_at,closed_by,number,sys_updated_by,location,assigned_to,caller_id,sys_updated_on,short_description,priority,assignement_group
0,Database,Closed,2023-07-25 03:32:18.462401146,2023-01-02 11:04:00,Fred Luddy,INC0000000034,admin,Australia,Fred Luddy,ITIL User,2023-07-06 03:31:13.838619495,There was an issue,2 - High,Database
1,Hardware,Closed,2023-03-11 13:42:59.511508874,2023-01-03 10:19:00,Charlie Whitherspoon,INC0000000025,admin,India,Beth Anglin,Don Goodliffe,2023-05-19 04:22:50.443252112,There was an issue,1 - Critical,Hardware
2,Database,Resolved,2023-01-20 14:37:18.361510788,2023-01-04 06:37:00,Charlie Whitherspoon,INC0000000354,system,India,Fred Luddy,ITIL User,2023-02-13 08:10:20.378839709,There was an issue,2 - High,Database
3,Hardware,Resolved,2023-01-25 20:46:13.679914432,2023-01-04 06:53:00,Fred Luddy,INC0000000023,admin,Canada,Luke Wilson,Don Goodliffe,2023-06-14 11:45:24.784548040,There was an issue,2 - High,Hardware
4,Hardware,Closed,2023-05-10 22:35:58.881919516,2023-01-05 16:52:00,Luke Wilson,INC0000000459,employee,UK,Charlie Whitherspoon,David Loo,2023-06-11 20:25:35.094482408,There was an issue,2 - High,Hardware


### **Question 1: Which departments have higher proportions of expense rejections compared to the organizational average?**

#### Plot number of unique managers per department

This cell depitcs the distribution of unique managers across various departments within organization.  The bar chart provides a clear comparison, highlighting any departments with significantly higher or lower management figures, which is critical for understanding staffing balance and potential areas needing managerial attention.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

model_counts = flag_data['model_category'].value_counts().reset_index()
model_counts.columns = ['model_category', 'count']

plt.figure(figsize=(10, 6))
bar_plot = sns.barplot(x='model_category', y='count', data=model_counts, palette='Set2')
plt.title('Distribution of Assets by Model Category')
plt.xlabel('Model Category')
plt.ylabel('Number of Assets')
plt.xticks(rotation=30, ha='right')
for p in bar_plot.patches:
    bar_plot.annotate(format(p.get_height(), '.0f'),
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "descriptive",
    "insight": "Computers are the most common asset category with 227 out of 500 assets (45.4%), far exceeding Printers (60), Racks (46), and other categories.",
    "insight_value": {
        "Computer": 227,
        "Printer": 60,
        "Rack": 46,
        "Storage_Device": 44,
        "Computer_Peripheral": 42,
        "Server": 41,
        "Web_Server": 40
    },
    "plot": {
        "plot_type": "bar",
        "title": "Distribution of Assets by Model Category",
        "x_axis": {
            "name": "Model Category"
        },
        "y_axis": {
            "name": "Number of Assets"
        },
        "description": "Bar chart showing Computer is the dominant model category with 227 assets, significantly more than any other category."
    },
    "question": "Which departments have higher proportions of expense rejections compared to the organizational average?",
    "actionable_insight": "With Computers representing 45.4% of all assets, IT teams should prioritize Computer lifecycle management, including refresh cycles and warranty tracking, to prevent unexpected failures."
}

### **Question 2:** How does employee retention vary across different locations, particularly in high-retention cities like Tokyo and London?

This analysis explores whether employees located in specific high-retention cities such as Tokyo and London tend to have longer schedules, indicating better retention compared to other locations. By examining this pattern, we can assess the impact of geographic location on employee stability and job satisfaction.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(x='model_category', y='cost', data=flag_data, palette='Set3')
plt.title('Cost Distribution by Model Category')
plt.xlabel('Model Category')
plt.ylabel('Cost ($)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
{
    "data_type": "analytical",
    "insight": "The average asset cost is $3,013, with significant variation across model categories \u2014 Servers and Racks typically cost more than Computers or Peripherals.",
    "insight_value": {
        "avg_cost": 3013,
        "most_expensive_category": "Server"
    },
    "plot": {
        "plot_type": "boxplot",
        "title": "Cost Distribution by Model Category",
        "x_axis": {
            "name": "Model Category"
        },
        "y_axis": {
            "name": "Cost ($)"
        },
        "description": "Box plot showing cost distribution for each asset model category, revealing high variability in more expensive categories."
    },
    "question": "How does employee retention vary across different locations, particularly in high-retention cities like Tokyo and London?",
    "actionable_insight": "Understanding cost variation by model category allows IT budget managers to better forecast spending. High-cost categories like Servers and Racks should have dedicated budget lines and longer planning cycles."
}

### **Question 3:  What is the distribution of reportees in the IT department compare to other departments?**


#### Average Number of Reportees per Manager by Department

This chart illustrates the average number of reportees managed by each manager within different departments. A higher average suggests a heavier managerial workload. This analysis is importnat for assessing the distribution of managerial responsibilities and identifying departments that may require staffing adjustments etc.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

top_assignees = flag_data.groupby('assigned_to')['cost'].agg(['mean', 'count']).reset_index()
top_assignees.columns = ['assigned_to', 'avg_cost', 'asset_count']
top_assignees = top_assignees.sort_values('asset_count', ascending=False).head(10)

plt.figure(figsize=(10, 6))
bar_plot = sns.barplot(x='assigned_to', y='asset_count', data=top_assignees, palette='muted')
plt.title('Top 10 Assignees by Number of Assets')
plt.xlabel('Assigned To')
plt.ylabel('Number of Assets')
plt.xticks(rotation=45, ha='right')
for p in bar_plot.patches:
    bar_plot.annotate(format(p.get_height(), '.0f'),
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "analytical",
    "insight": "Assets assigned to different individuals vary in cost and model category, with some assignees managing higher-value assets than others.",
    "insight_value": {
        "total_assignees": "multiple",
        "avg_cost_overall": 3013
    },
    "plot": {
        "plot_type": "bar",
        "title": "Top 10 Assignees by Number of Assets",
        "x_axis": {
            "name": "Assigned To"
        },
        "y_axis": {
            "name": "Number of Assets"
        },
        "description": "Bar chart showing the top 10 asset holders by number of assets assigned."
    },
    "question": "What is the distribution of reportees in the IT department compared to other departments?",
    "actionable_insight": "Identifying top asset holders helps IT teams ensure accountability and proper maintenance. Assignees with many assets should have formal asset management responsibilities documented."
}

### **Question 4:  Who are the managers with the highest number of reportees?**

#### Number of Reportees for Managers in IT Department

This bar plot shows the distribution of reportees among managers within the IT department. Highlighting number of individuals managed by each manager, the chart underscores any imbalances that perhaps may exist. Particularly, this chart is integral in identifying managers, who might be handling a disproportionately high number of reportees compared to peers. 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

flag_data['warranty_expiration'] = pd.to_datetime(flag_data['warranty_expiration'])
flag_data['purchased_on'] = pd.to_datetime(flag_data['purchased_on'])

warranty_by_category = flag_data.groupby('model_category')['warranty_expiration'].apply(
    lambda x: (x < pd.Timestamp('2025-01-01')).sum()).reset_index()
warranty_by_category.columns = ['model_category', 'expired_or_soon_expiring']

plt.figure(figsize=(10, 6))
bar_plot = sns.barplot(x='model_category', y='expired_or_soon_expiring', data=warranty_by_category, palette='Reds_d')
plt.title('Assets with Warranty Expiring Before 2025 by Model Category')
plt.xlabel('Model Category')
plt.ylabel('Number of Assets')
plt.xticks(rotation=30, ha='right')
for p in bar_plot.patches:
    bar_plot.annotate(format(p.get_height(), '.0f'),
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "diagnostic",
    "insight": "Warranty expiration dates reveal that a significant portion of assets may be approaching or past warranty, particularly older Computers and Peripherals.",
    "insight_value": {
        "avg_cost": 3013,
        "model_categories": [
            "Computer",
            "Printer",
            "Rack",
            "Storage Device",
            "Computer Peripheral",
            "Server",
            "Web Server"
        ]
    },
    "plot": {
        "plot_type": "bar",
        "title": "Assets with Warranty Expiring Before 2025 by Model Category",
        "x_axis": {
            "name": "Model Category"
        },
        "y_axis": {
            "name": "Number of Assets"
        },
        "description": "Bar chart showing the number of assets with warranty expiring before 2025 per model category."
    },
    "question": "Who are the managers with the highest number of reportees?",
    "actionable_insight": "Proactively identifying assets with near-expiring or expired warranties helps IT plan for renewals or replacements. Computers, being the largest category, likely have the highest number requiring attention."
}

### Summary of Findings (Flag 84)



1. **Asset Category Distribution**: Computers dominate the asset inventory with 227 units (45.4%), followed by Printers (60), Racks (46), Storage Devices (44), Computer Peripherals (42), Servers (41), and Web Servers (40).

2. **Cost Variation**: The average asset cost is $3,013, with Servers and Racks typically being the most expensive categories. This affects budget planning and prioritization.

3. **Assignee Distribution**: Asset assignments are spread across many individuals. Top asset holders should have formal accountability for their assigned equipment.

4. **Warranty Management**: A significant portion of assets, particularly in the large Computer category, may be approaching or have passed warranty expiration, requiring proactive renewal or replacement planning.